In [1]:
import os
import chromadb 
import numpy as np
import polars as pl

from typing import *
from chromadb import Client
from tqdm.notebook import tqdm
from chromadb.utils import embedding_functions as ef
from datasets import Dataset, Features, Sequence, Value, load_from_disk

In [2]:
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_float("full")

polars.config.Config

In [3]:
downstream_idx = pl.read_parquet('./downstream_idx.parquet')

# Tools

In [4]:
import os
import json
import torch
import bisect

import polars as pl

from math import ceil
from pathlib import Path
from collections import defaultdict
from datasets import load_from_disk
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from typing import Dict, Iterable, List, Optional, Any, Union, Literal, Tuple

In [5]:
class Tokenizer:
    def __init__(
        self,
        codes_parquet_fp: str,
        special_tokens: Optional[Iterable[str]] = None,
        force_special_ids: bool = True,  # pin [PAD]=0 etc.
    ):
        if special_tokens is None:
            special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]


        df_codes = pl.read_parquet(str(codes_parquet_fp), columns=["code"])
        base_codes = df_codes.get_column("code").to_list()
        seen = set()
        unique_codes = []
        for c in base_codes:
            if c not in seen:
                unique_codes.append(c)
                seen.add(c)

        vocab_list: List[str] = []
        special_tokens = list(special_tokens)

        if force_special_ids:
            for tok in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]:
                if tok in special_tokens and tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
                elif tok in special_tokens and tok in seen:

                    vocab_list.append(tok)
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
        else:
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)


        vocab_list.extend(unique_codes)


        self.id2code: List[str] = vocab_list
        self.code2id: Dict[str, int] = {tok: idx for idx, tok in enumerate(vocab_list)}
        self.vocab_size: int = len(self.id2code)

        self.pad_token  = "[PAD]" if "[PAD]" in self.code2id else None
        self.mask_token = "[MASK]" if "[MASK]" in self.code2id else None
        self.cls_token  = "[CLS]" if "[CLS]" in self.code2id else None
        self.unk_token  = "[UNK]" if "[UNK]" in self.code2id else None

        self.pad_id  = self.code2id[self.pad_token]  if self.pad_token  else 0
        self.mask_id = self.code2id[self.mask_token] if self.mask_token else None
        self.cls_id  = self.code2id[self.cls_token]  if self.cls_token  else None
        self.unk_id  = self.code2id[self.unk_token]  if self.unk_token  else None


        type_set = set()
        for tok in self.id2code:
            prefix = tok.split("//", 1)[0]
            type_set.add(prefix)

        types_sorted = sorted(t for t in type_set if t not in ("[PAD]",))
        self.type2id: Dict[str, int] = {"[PAD]": 0}
        next_id = 1
        for sp in ["[MASK]", "[CLS]", "[UNK]"]:
            if sp in type_set:
                self.type2id[sp] = next_id; next_id += 1
        for t in types_sorted:
            if t not in self.type2id:
                self.type2id[t] = next_id
                next_id += 1

        self._code2id_df = pl.DataFrame({"code": self.id2code,
                                         "input_id": list(range(self.vocab_size))}) \
                               .with_columns(pl.col("code").cast(pl.Categorical))
        self._type2id_df = pl.DataFrame({"code_type": list(self.type2id.keys()),
                                         "type_id":   list(self.type2id.values())}) \
                               .with_columns(pl.col("code_type").cast(pl.Categorical))


    def encode(self, codes: Iterable[str]) -> List[int]:
        get = self.code2id.get
        if self.unk_id is not None:
            fallback = self.unk_id
        else:
            fallback = self.pad_id if self.pad_id is not None else 0
        return [get(c, fallback) for c in codes]

    def decode(self, ids: Iterable[int]) -> List[str]:
        out = []
        for i in ids:
            if 0 <= i < self.vocab_size:
                out.append(self.id2code[i])
            else:
                out.append(self.unk_token or "[UNK]")
        return out

    def save(self, path: str) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        obj = {
            "id2code": self.id2code,
            "special_tokens": [t for t in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"] if t in self.code2id],
            "type2id": self.type2id,
            "pad_id": self.pad_id,
            "mask_id": self.mask_id,
            "cls_id": self.cls_id,
            "unk_id": self.unk_id,
        }
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)

    @classmethod
    def load(cls, path: str) -> "Tokenizer":
        path = Path(path)
        with open(path) as f:
            obj = json.load(f)

        tok = cls.__new__(cls) 

        tok.id2code = obj["id2code"]
        tok.code2id = {tok_: i for i, tok_ in enumerate(tok.id2code)}
        tok.vocab_size = len(tok.id2code)

        tok.special_tokens = obj.get("special_tokens", [])
        tok.type2id = obj.get("type2id", {})

        tok.pad_token  = "[PAD]" if "[PAD]" in tok.code2id else None
        tok.mask_token = "[MASK]" if "[MASK]" in tok.code2id else None
        tok.cls_token  = "[CLS]" if "[CLS]" in tok.code2id else None
        tok.unk_token  = "[UNK]" if "[UNK]" in tok.code2id else None

        tok.pad_id  = obj.get("pad_id", tok.code2id.get("[PAD]", 0))
        tok.mask_id = obj.get("mask_id", tok.code2id.get("[MASK]")) if "[MASK]" in tok.code2id else None
        tok.cls_id  = obj.get("cls_id", tok.code2id.get("[CLS]"))   if "[CLS]" in tok.code2id else None
        tok.unk_id  = obj.get("unk_id", tok.code2id.get("[UNK]"))   if "[UNK]" in tok.code2id else None

        # Rebuild Polars lookup frames
        tok._code2id_df = pl.DataFrame({"code": tok.id2code,
                                        "input_id": list(range(tok.vocab_size))}) \
                              .with_columns(pl.col("code").cast(pl.Categorical))
        tok._type2id_df = pl.DataFrame({"code_type": list(tok.type2id.keys()),
                                        "type_id":   list(tok.type2id.values())}) \
                              .with_columns(pl.col("code_type").cast(pl.Categorical))
        return tok

    @property
    def code2id_df(self) -> pl.DataFrame:
        return self._code2id_df

    @property
    def type2id_df(self) -> pl.DataFrame:
        return self._type2id_df

In [6]:
class SequencesGenerator:


    def __init__(
        self,
        tokenizer_path: str,
        chunk_length: int = 1024,
        overlap: int = 128,
        return_numeric: bool = False,
        return_text: bool = False,
    ):

        self.tokenizer = Tokenizer.load(tokenizer_path)
        self.chunk_length = chunk_length
        self.overlap = overlap
        self.return_numeric = return_numeric
        self.return_text = return_text

    def encode_sequence(
        self,
        timeline: pl.DataFrame,
        max_length: Optional[int] = None,
        pad_to_max: bool = False,
        truncation: Literal["head", "tail"] = "tail",
        add_cls: bool = False,
    ) -> Dict[str, Union[List[int], List[float], List[str]]]:
        """
        Vectorized build of:
          input_ids, attention_mask, visit_ids, stage_ids, type_ids
          + optional numeric/text streams (+ masks)
        """
        df = timeline

        # --- visit_ids (dense per-timeline integer ids starting at 1) ---
        # If seq_id exists -> categorical codes as stable ints; otherwise zeros.
        if "seq_id" in df.columns:
        # unique stable integer ids starting at 1
            uniq = df.select(pl.col("seq_id")).unique(maintain_order=True)
            uniq = uniq.with_row_count(name="visit_ids_raw")  # 0..K-1
            df = df.join(uniq, on="seq_id", how="left").with_columns(
                (pl.col("visit_ids_raw") ).alias("visit_id").fill_null(0)
            ).drop("visit_ids_raw")
        else:
            df = df.with_columns(pl.lit(0).alias("visit_id"))

        # --- stage_ids (first non-null of stage cols) ---
        stage_cols = ["out_id", "er_id", "hadm_id", "icustay_id"]
        present_stages = [c for c in stage_cols if c in df.columns]
        if present_stages:
            # Compute stage_id = 1..len(present_stages) or 0 if all null
            # We scan in the stage_cols order and pick the first non-null
            expr = pl.lit(0)
            for i, col in enumerate(present_stages, start=1):
                expr = pl.when(expr.eq(0) & pl.col(col).is_not_null()).then(i).otherwise(expr)
            df = df.with_columns(expr.alias("stage_id"))
        else:
            df = df.with_columns(pl.lit(0).alias("stage_id"))

        # --- map code_type -> type_id (PAD type=0 already in tokenizer) ---
        df = df.join(
            self.tokenizer.type2id_df,
            on=pl.col("code_type").cast(pl.Categorical),
            how="left",
        ).with_columns(pl.col("type_id").fill_null(0))

        # --- map code -> input_id (PAD=0, UNK fallback handled downstream) ---
        df = df.join(
            self.tokenizer.code2id_df,
            on=pl.col("code").cast(pl.Categorical),
            how="left",
        )

        # Fill unknown codes to UNK (or PAD if no UNK)
        unk_id = self.tokenizer.unk_id if self.tokenizer.unk_id is not None else self.tokenizer.pad_id or 0
        df = df.with_columns(pl.col("input_id").fill_null(unk_id))

        # --- TIME-GAP tokens zero out visit_id & stage_id (patient-agnostic) ---
        df = df.with_columns(
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("visit_id"))
              .alias("visit_id"),
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("stage_id"))
              .alias("stage_id"),
        )

        # --- Add [CLS] row if requested ---
        if add_cls and (self.tokenizer.cls_token is not None):
            cls_row = {
                "code": self.tokenizer.cls_token,
                "code_type": "[CLS]",
                "visit_id": 0,
                "stage_id": 0,
                "type_id": self.tokenizer.type2id.get("[CLS]", 0),
                "input_id": self.tokenizer.cls_id,
            }
            # Optional numeric/text placeholders
            if self.return_numeric:
                cls_row["numeric_value"] = None
            if self.return_text:
                cls_row["text_value"] = None

            df = pl.concat([pl.DataFrame([cls_row]), df], how="vertical_relaxed")

        # --- extract arrays ---
        input_ids = df.get_column("input_id").cast(pl.Int64).to_list()
        type_ids = df.get_column("type_id").cast(pl.Int64).to_list()
        visit_ids = df.get_column("visit_id").cast(pl.Int64).to_list()
        stage_ids = df.get_column("stage_id").cast(pl.Int64).to_list()
        attention_mask = [1] * len(input_ids)

        # --- optional value streams (numeric/text) ---
        value_payload = self._build_value_streams(
            df=df,
            max_length=max_length,
            pad_to_max=pad_to_max,
            truncation=truncation,
        )

        # --- truncate/pad core streams in one go ---
        input_ids      = self._truncate(input_ids,      max_length, truncation)
        type_ids       = self._truncate(type_ids,       max_length, truncation)
        visit_ids      = self._truncate(visit_ids,      max_length, truncation)
        stage_ids      = self._truncate(stage_ids,      max_length, truncation)
        attention_mask = [1] * len(input_ids)

        if pad_to_max and max_length is not None and len(input_ids) < max_length:
            pad_len = max_length - len(input_ids)
            pad_id = self.tokenizer.pad_id if self.tokenizer.pad_id is not None else 0
            input_ids      = input_ids + [pad_id] * pad_len
            type_ids       = type_ids + [0] * pad_len
            visit_ids      = visit_ids + [0] * pad_len
            stage_ids      = stage_ids + [0] * pad_len
            attention_mask = attention_mask + [0] * pad_len

        out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "visit_ids": visit_ids,
            "stage_ids": stage_ids,
            "type_ids": type_ids,
        }
        out.update(value_payload)
        return out

    def get_overlapped_chunks(
        self,
        timeline: Dict[str, Iterable],
        chunk_length: Optional[int] = None,
        overlap: Optional[int] = None,
        add_cls_per_chunk: bool = True,
    ) -> List[Dict[str, List[Any]]]:
        """
        Sliding-window chunking with optional [CLS] per chunk and padding.
        """
        if chunk_length is None or overlap is None:
            chunk_length = self.chunk_length
            overlap = self.overlap

        fields = ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"]
        for extra in ("numeric_values", "numeric_mask", "text_values", "text_mask"):
            if extra in timeline and extra not in fields:
                fields.append(extra)

        n = len(timeline["input_ids"])
        payload = chunk_length - (1 if add_cls_per_chunk else 0)
        step = max(1, payload - overlap)

        num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
        starts = [i * step for i in range(num_chunks)]

        cls_defaults = {
            "input_ids": self.tokenizer.cls_id if self.tokenizer.cls_id is not None else (self.tokenizer.pad_id or 0),
            "attention_mask": 1,
            "visit_ids": 0,
            "stage_ids": 0,
            "type_ids": self.tokenizer.type2id.get("[CLS]", 0),
            "numeric_values": 0.0,
            "numeric_mask": 0,
            "text_values": "",
            "text_mask": 0,
        }

        chunks = []
        for start in starts:
            end = min(n, start + payload)
            sliced = {k: list(timeline[k][start:end]) for k in fields if k in timeline}

            if "attention_mask" in sliced:
                sliced["attention_mask"] = [1] * len(sliced["input_ids"])

            if add_cls_per_chunk:
                for k in list(sliced.keys()):
                    sliced[k] = [cls_defaults[k]] + sliced[k]

            cur_len = len(sliced["input_ids"])
            if cur_len < chunk_length:
                pad_len = chunk_length - cur_len
                for k in list(sliced.keys()):
                    sliced[k] = self._pad_list(sliced[k], pad_len, 0)

            chunks.append(sliced)
        return chunks


    def _build_value_streams(
        self,
        df: pl.DataFrame,
        max_length: Optional[int],
        pad_to_max: bool,
        truncation: Literal["head", "tail"],
    ) -> Dict[str, List[Any]]:
        out: Dict[str, List[Any]] = {}
        # Numeric stream
        if self.return_numeric:
            if "numeric_value" in df.columns:
                vals = df.get_column("numeric_value").to_list()
            else:
                vals = [None] * df.height
            num_mask = [1 if (v is not None) else 0 for v in vals]
            vals = [0.0 if v is None else float(v) for v in vals]

            vals = self._truncate(vals, max_length, truncation)
            num_mask = self._truncate(num_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(vals) < max_length:
                pad_len = max_length - len(vals)
                vals += [0.0] * pad_len
                num_mask += [0] * pad_len

            out["numeric_values"] = vals
            out["numeric_mask"] = num_mask

        # Text stream
        if self.return_text:
            if "text_value" in df.columns:
                txt = df.get_column("text_value").to_list()
            else:
                txt = [None] * df.height
            txt = [("" if (t is None or str(t) == "___") else str(t)) for t in txt]
            txt_mask = [1 if (t != "") else 0 for t in txt]

            txt = self._truncate(txt, max_length, truncation)
            txt_mask = self._truncate(txt_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(txt) < max_length:
                pad_len = max_length - len(txt)
                txt += [""] * pad_len
                txt_mask += [0] * pad_len

            out["text_values"] = txt
            out["text_mask"] = txt_mask

        return out

    @staticmethod
    def _truncate(seq: List[Any], max_length: Optional[int], truncation: str) -> List[Any]:
        if max_length is None or len(seq) <= max_length:
            return seq
        return seq[-max_length:] if truncation == "head" else seq[:max_length]

    @staticmethod
    def _pad_list(lst: List[Any], pad_len: int, pad_value: Any) -> List[Any]:
        if pad_len <= 0:
            return lst
        return lst + [pad_value] * pad_len

In [7]:
limits = {
    'within24_query': {512:  ['w24_start_512',  'w24_end_512' ],
                       1024: ['w24_start_1024', 'w24_end_1024'],
                       1536: ['w24_start_1536', 'w24_end_1536'],
                      },
    
    'within24_hist_icu': {512: ['wStay_min', 'w24_start_512' ],
                         1024: ['wStay_min', 'w24_start_1024'],
                         1536: ['wStay_min', 'w24_start_1536'],
                      },
    
    'within24_hist_full': {512: [ 0, 'w24_start_512' ],
                          1024: [ 0, 'w24_start_1024'],
                          1536: [ 0, 'w24_start_1536'],
                          },

    
    'within48_query': {512:  ['w48_start_512',  'w48_end_512' ],
                       1024: ['w48_start_1024', 'w48_end_1024'],
                       1536: ['w48_start_1536', 'w48_end_1536'],
                      },

    'within48_hist_icu': {512: ['wStay_min', 'w48_start_512' ],
                         1024: ['wStay_min', 'w48_start_1024'],
                         1536: ['wStay_min', 'w48_start_1536']
                      },
    
    'within48_hist_full': {512: [ 0, 'w48_start_512' ],
                          1024: [ 0, 'w48_start_1024'],
                          1536: [ 0, 'w48_start_1536']
                          },
    
    'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ],
                          1024: ['wStay_start_1024', 'wStay_end_1024'],
                          1536: ['wStay_start_1536', 'wStay_end_1536']
                         },
    
    'within_stay_hist_icu': {512:  ['wStay_min', 'wStay_start_512' ],
                             1024: ['wStay_min', 'wStay_start_1024'],
                             1536: ['wStay_min', 'wStay_start_1536'],
                            },
    
    'within_stay_hist_full': {512:  [ 0, 'w48_start_512' ],
                              1024: [ 0, 'w48_start_1024'],
                              1536: [ 0, 'w48_start_1536'],
                             },
    }

# Embedding Function

In [8]:
import torch
import torch.nn as nn
from transformers import RoFormerModel, RoFormerConfig  

In [9]:
# IMPORTANT
MAX_VISITS = 101 
VOCAB_SIZE = 47377
config = RoFormerConfig(vocab_size=VOCAB_SIZE,
                        hidden_size=768,
                        num_hidden_layers=12,
                        num_attention_heads=12,
                        intermediate_size=3072,
                        max_position_embeddings=512,
                        pad_token_id=0,
                        type_vocab_size= 28,
                        visit_vocab_size= 102,
                        stage_vocab_size= 5)

In [10]:
class EHREmbeddings(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_size: int,
        pad_token_id: int = 0,
        type_vocab_size: int = 28,
        visit_vocab_size: int = 102,
        stage_vocab_size: int = 5,
        dropout: float = 0.1
    ):
        super().__init__()
        self.tok_emb   = nn.Embedding(vocab_size, embedding_size, padding_idx=pad_token_id)
        self.type_emb  = nn.Embedding(type_vocab_size,  embedding_size, padding_idx=pad_token_id)
        self.visit_emb = nn.Embedding(visit_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.stage_emb = nn.Embedding(stage_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.norm      = nn.LayerNorm(embedding_size)
        self.drop      = nn.Dropout(dropout)

    def encode(self, input_ids, type_ids, visit_ids, stage_ids):
        x = self.tok_emb(input_ids.long())
        x = x + self.type_emb(type_ids.long())
        x = x + self.visit_emb(visit_ids.long())
        x = x + self.stage_emb(stage_ids.long())
        return self.drop(self.norm(x))


    def forward(self, input_ids=None, 
                token_type_ids=None, 
                inputs_embeds=None,
                **kwargs):
        if inputs_embeds is not None:
            return inputs_embeds
        return self.drop(self.norm(self.tok_emb(input_ids.long())))

In [11]:
class RoformerEHREmbedder(nn.Module):

    def __init__(
        self,
        config,
        dropout: float = 0.1,
        ckpt_path: str = None,   
        pool: str = "cls",              
        attn_pool_dim: int = 128,       
        normalize: bool = False,         
        device: str = "cuda" if torch.cuda.is_available() else "cpu",
        dtype: torch.dtype = torch.float32
    ):
        super().__init__()
        self.config = config
        self.pool = pool
        self.normalize = normalize
        self.device_ = device
        self.dtype_ = dtype

        # backbone without classification head
        self.backbone = RoFormerModel(self.config)

        # plug in your custom input embedding layer
        self.backbone.embeddings = EHREmbeddings(
            vocab_size=config.vocab_size,
            embedding_size=config.embedding_size,
            pad_token_id=config.pad_token_id,
            type_vocab_size=config.type_vocab_size,
            visit_vocab_size=config.visit_vocab_size,
            stage_vocab_size=config.stage_vocab_size,
            dropout=dropout
        )

        # optional attentive pooling
#         if self.pool == "attn":
#             self.attn_pool = nn.Sequential(
#                 nn.Linear(self.config.hidden_size, attn_pool_dim, bias=True),
#                 nn.Tanh(),
#                 nn.Linear(attn_pool_dim, 1, bias=False)
#             )

        # load weights (either full HF state_dict or your Lightning ckpt)
        if ckpt_path:
            self.get_pretrained_weights(backbone= self.backbone, ckpt_path = ckpt_path)

        self.eval().to(self.device_, dtype=self.dtype_)

    @torch.no_grad()
    def encode(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        type_ids: torch.Tensor,
        visit_ids: torch.Tensor,
        stage_ids: torch.Tensor,
        pad_token_id: int = 0
    ) -> torch.Tensor:
        """
        Returns [B, D] embeddings.
        """
        # Build inputs_embeds via your EHREmbeddings.encode
        inputs_embeds = self.backbone.embeddings.encode(
            input_ids=input_ids.to(self.device_),
            type_ids=type_ids.to(self.device_),
            visit_ids=visit_ids.to(self.device_),
            stage_ids=stage_ids.to(self.device_)
        ).to(self.dtype_)

        outputs = self.backbone(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask.to(self.device_)
        )
        # candidates: last_hidden_state, pooled_output (if available)
        last_hidden = outputs.last_hidden_state  # [B, L, H]

        if self.pool == "cls":
            vec = last_hidden[:, 0, :]  # [CLS]-style
        elif self.pool == "mean":
            # mask-aware mean pooling
            mask = attention_mask.unsqueeze(-1).to(last_hidden.dtype)  # [B,L,1]
            vec = (last_hidden * mask).sum(dim=1) / (mask.sum(dim=1).clamp(min=1.0))
#         else:  # attn
#             scores = self.attn_pool(last_hidden).squeeze(-1)         # [B,L]
#             scores = scores.masked_fill(attention_mask == 0, -1e9)
#             w = torch.softmax(scores, dim=-1).unsqueeze(-1)          # [B,L,1]
#             vec = (last_hidden * w).sum(dim=1)                        # [B,H]

        if self.normalize:
            vec = nn.functional.normalize(vec, p=2, dim=-1)
        return vec  # [B, H]
    
    def get_pretrained_weights(self,
                               backbone: nn.Module,
                               ckpt_path: str):
        sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        if isinstance(sd, dict) and "state_dict" in sd:
            sd = sd["state_dict"]

        new_sd = {}
        for k, v in sd.items():
            # drop non-backbone heads
            if k.startswith(("cls.", "lm_head.", "classifier.", "score.")):
                continue
            # strip leading "backbone."
            if k.startswith("backbone."):
                k = k[len("backbone."):]
            # map "roformer." -> "" (RoFormerModel params live at root)
            if k.startswith("roformer."):
                k = k[len("roformer."):]
            new_sd[k] = v

        # keep only keys that actually exist in target
        target_keys = set(backbone.state_dict().keys())
        filtered = {k: v for k, v in new_sd.items() if k in target_keys}

        print(f"keeping {len(filtered)}/{len(new_sd)} keys")
        missing, unexpected = backbone.load_state_dict(filtered, strict=False)
        print("Loaded with:", {"missing": missing, "unexpected": unexpected})

In [12]:
class EmbedCollator:
    def __init__(self) -> None:
        pass


    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        chunks = self._flatten(batch)
        out = self._stack(chunks)
        
        return out


    def _flatten(self, batch) -> List[Dict]:
        out: List[Dict] = []
        for item in batch:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, (list, tuple)):
                out.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")
        if not out:
            raise ValueError("Empty batch after normalization.")
        return out

    def _stack(self, chunks: List[Dict]) -> Dict[str, torch.Tensor]:
        keys = list(chunks[0].keys())
        out = {}
        for k in keys:
            # Skip known non-numeric or variable-shaped fields
            if k in ("text_values",):  # add others you don't want to collate
                continue

            # Replace Nones with safe defaults
            seq_list = []
            for c in chunks:
                v = c[k]
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]  # 0 for ints/floats
                elif v is None:
                    # single value case (shouldn't happen for sequences, but guard anyway)
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)
        return out

In [13]:
collate_fn = EmbedCollator()

In [14]:
from chromadb.api.types import Documents, Embeddings, EmbeddingFunction
from typing import Dict, List, Optional, Union
import torch, json

class ChromaEHREmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, embedder, collate_fn, batch_size: Optional[int] = None):
        self.embedder = embedder
        self.collate_fn = collate_fn
        self.batch_size = batch_size  # None => all at once

    def _decode_docs(self, docs: Documents):
        """Accept raw dicts or JSON strings; preserve list-of-lists (patients→chunks)."""
        out: List[Union[dict, list]] = []
        for item in docs:
            if isinstance(item, str):
                out.append(json.loads(item))                     # JSON → dict
            elif isinstance(item, (list, tuple)):
                out.append([json.loads(x) if isinstance(x, str) else x for x in item])
            else:
                out.append(item)                                 # dict
        return out

    def _slice_batch(self, batch_tensors: Dict[str, torch.Tensor], sl: slice) -> Dict[str, torch.Tensor]:
        keys = ("input_ids", "attention_mask", "type_ids", "visit_ids", "stage_ids")
        return {k: batch_tensors[k][sl] for k in keys if k in batch_tensors}

    def __call__(self, docs: Documents) -> Embeddings:
        # 1) Handle both cases: dicts (upsert) or JSON strings (retrieval)
        docs = self._decode_docs(docs)

        # 2) Collate to tensors
        batch_inputs = self.collate_fn(docs)  # -> dict of tensors [B, L]

        # 3) Device/dtype
        device = getattr(self.embedder, "device_", "cpu")
        for k in ("input_ids", "attention_mask", "type_ids", "visit_ids", "stage_ids"):
            if k not in batch_inputs:
                raise KeyError(f"Missing required key '{k}' in collated batch.")
            batch_inputs[k] = batch_inputs[k].to(device=device, dtype=torch.long)

        # 4) Encode (optional mini-batching)
        B = batch_inputs["input_ids"].size(0)
        bs = self.batch_size or B

        embs_out: List[torch.Tensor] = []
        with torch.no_grad():
            for start in range(0, B, bs):
                end = min(start + bs, B)
                sub = self._slice_batch(batch_inputs, slice(start, end))
                v = self.embedder.encode(
                    input_ids=sub["input_ids"],
                    attention_mask=sub["attention_mask"],
                    type_ids=sub["type_ids"],
                    visit_ids=sub["visit_ids"],
                    stage_ids=sub["stage_ids"],
                )
                embs_out.append(v.detach().cpu())

        return torch.cat(embs_out, dim=0).tolist()

In [15]:
embedder = RoformerEHREmbedder(
    config=config,
    ckpt_path="/scratch/sas10092/ehr-foundation/models/mlm/wandb/run-20251102_081006-Roformer_base_12710830_512_64_25_maskprob_12.5overlap/files/ckpt/epoch=69-step=631120.ckpt",
    pool="cls",                        
    normalize=False)

collate_fn = EmbedCollator()

ehr_embedf = ChromaEHREmbeddingFunction(embedder=embedder, collate_fn=collate_fn, batch_size=None)

keeping 199/206 keys
Loaded with: {'missing': [], 'unexpected': []}


In [16]:
import json
import numpy as np
import polars as pl
from tqdm import tqdm
from collections import defaultdict
from datasets import load_from_disk
import chromadb




class VectorDBUploader:
    def __init__(self,
                 collection,
                 seq_gen: SequencesGenerator,
                 main_window: str,
                 seq_length: int,
                 limits: dict,
                 data_idx_path: str,
                 data_path: str,
                 needed_cols: list = ['subject_id', 'input_ids', 'attention_mask', 'visit_ids', 'stage_ids', 'type_ids']
                 ):
        # Initialize client and collection

        self.collection = collection
        # Initialize sequence generator
        self.seq_gen = seq_gen

        # Store parameters
        self.main_window = main_window
        self.seq_length = seq_length
        self.limits = limits
        self.needed_cols = needed_cols

        # Load index and dataset
        self.data_idx = pl.read_parquet(data_idx_path)
        sub_ids = set(self.data_idx.get_column("subject_id").to_list())
        hf_dataset = load_from_disk(data_path)

        # Filter dataset to match index subjects
        hf_dataset = hf_dataset.filter(
            lambda sids: [sid in sub_ids for sid in sids],
            batched=True,
            input_columns="subject_id",
        )

        # Format and subset dataset
        hf_dataset = (
            hf_dataset
            .flatten_indices()
            .select_columns(self.needed_cols)
            .with_format("numpy", columns=self.needed_cols, output_all_columns=False)
        )

        self.hf_dataset = hf_dataset
        self.index = self._build_index()

    def _build_index(self):
        """Build subject_id → list of indices mapping."""
        sids = self.hf_dataset["subject_id"]
        index = defaultdict(list)
        for i, sid in enumerate(sids):
            index[sid].append(i)
        return index

    def upsert_chunks(self):
        """Iterate over data_idx and upload encoded chunks to ChromaDB."""
        for i in tqdm(range(len(self.data_idx))):
            stay = self.data_idx[i]
            split = stay['split'][0]
            subject_id = stay['subject_id'][0]
            start = self.limits[self.main_window][self.seq_length][0]
            end = stay[self.limits[self.main_window][self.seq_length][1]][0]
            timeline_encoded = self.hf_dataset.select(self.index[subject_id])[0]

            # Slice history window
            history_window = {
                k: (v[start:end].tolist() if isinstance(v, (list, np.ndarray)) else v)
                for k, v in timeline_encoded.items()
            }

            # Generate chunks and metadata
            chunks = self.seq_gen.get_overlapped_chunks(history_window)
            docs = [json.dumps(ch, separators=(",", ":")) for ch in chunks]
            ids = [f"{subject_id}__{i:05d}" for i in range(len(chunks))]
            metas = [{"subject_id": subject_id, "chunk_idx": i, "split": split} for i in range(len(chunks))]

            # Upload to Chroma
            self.collection.upsert(ids=ids, documents=docs, metadatas=metas)

In [24]:
# collection = clinet.get_collection(
#             name='within48_hist_full',
#             embedding_function=ehr_embedf
#         )

seq_gen = SequencesGenerator(
            tokenizer_path="../vocab.json",
            chunk_length=512,
            overlap=64,
            return_numeric=False,
            return_text=False
        )


uploader = VectorDBUploader(
    collection=collection,
    seq_gen=seq_gen,
    main_window='within24_hist_full',
    seq_length=512,
    limits=limits,
    data_idx_path="../downstream_idx.parquet",
    data_path="../data/meds_normalized_arrow/",
)





Loading dataset from disk:   0%|          | 0/32 [00:00<?, ?it/s]

In [50]:
# uploader.upsert_chunks()
# client.delete_collection('within24_hist_full')

In [31]:
# query chunk
from time import time 
tic = time()
patient = uploader.hf_dataset[8]
start = 17858
end = 18369

prediction_window = {k: (v[start:end].tolist() if isinstance(v, (np.ndarray)) else v) for k, v in patient.items()}
# prediction_window = {k: (v.tolist() if isinstance(v, (np.ndarray)) else v) for k, v in patient.items()}



q_chunk = seq_gen.get_overlapped_chunks(prediction_window)
q_json  = json.dumps(q_chunk, separators=(",", ":"))


subject_id = (
    int(patient["subject_id"][0]) if isinstance(patient["subject_id"], (list, tuple))
    else int(patient["subject_id"])
)

# Query
res = uploader.collection.query(
    query_texts=[q_json],
    n_results=20,
    where={"subject_id": subject_id},   
    include=["metadatas", "documents", "distances", ]
)
toc = time()
print(f'total time = {toc-tic}')
# Inspect
print("Top IDs:", res["ids"][0])
print('+'*100,'\n')
print("Distances:", res["distances"][0])
print('+'*100,'\n')
print("Top-1 metadata:", res["metadatas"][0])


total time = 0.6469466686248779
Top IDs: ['10002428__00005', '10002428__00006', '10002428__00022', '10002428__00014', '10002428__00007', '10002428__00044', '10002428__00028', '10002428__00040', '10002428__00024', '10002428__00009', '10002428__00026', '10002428__00041', '10002428__00042', '10002428__00031', '10002428__00046', '10002428__00004', '10002428__00019', '10002428__00030', '10002428__00049', '10002428__00023']
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ 

Distances: [0.08337998390197754, 0.10203206539154053, 0.10505670309066772, 0.116851806640625, 0.12714409828186035, 0.12927871942520142, 0.12942743301391602, 0.13809025287628174, 0.1384173035621643, 0.14218533039093018, 0.1435832977294922, 0.14871150255203247, 0.14949434995651245, 0.1495034098625183, 0.14989298582077026, 0.15260320901870728, 0.1541181206703186, 0.15472841262817383, 0.15906274318695068, 0.15956735610961914]
+++++++++++++++++++++++++++++++++++++++++++++++++

In [57]:
18369 -511

17858

# Retrieval pipelines

In [18]:
clinets_path = '../data/vectordbs/'
client_type = 'cls'
clients = os.path.join(clinets_path,client_type)
clients_list = os.listdir(clients)

In [19]:
clients_list

['within48_hist_full_1024_128',
 'within48_hist_full_512_64',
 'within48_hist_full_1536_192',
 'within_stay_hist_full_1536_192',
 'within_stay_hist_full_512_64',
 'within24_hist_full_1024_128',
 'within24_hist_full_512_64',
 'within24_hist_full_1536_192',
 'within_stay_hist_full_1024_128']

In [20]:
clinet = chromadb.PersistentClient(os.path.join(clients,'within48_hist_full_512_64'))

In [23]:
collection = clinet.get_collection(name='within48_hist_full',
                                 embedding_function=ehr_embedf
                                  )

In [40]:
from datasets import load_from_disk
from collections import defaultdict

In [41]:
limits = {
    'within24_query': {512:  ['w24_start_512',  'w24_end_512' ],
                       1024: ['w24_start_1024', 'w24_end_1024'],
                       1536: ['w24_start_1536', 'w24_end_1536'],
                      },
    
    'within24_hist_icu': {512: ['wStay_min', 'w24_start_512' ],
                         1024: ['wStay_min', 'w24_start_1024'],
                         1536: ['wStay_min', 'w24_start_1536'],
                      },
    
    'within24_hist_full': {512: [ 0, 'w24_start_512' ],
                          1024: [ 0, 'w24_start_1024'],
                          1536: [ 0, 'w24_start_1536'],
                          },

    
    'within48_query': {512:  ['w48_start_512',  'w48_end_512' ],
                       1024: ['w48_start_1024', 'w48_end_1024'],
                       1536: ['w48_start_1536', 'w48_end_1536'],
                      },

    'within48_hist_icu': {512: ['wStay_min', 'w48_start_512' ],
                         1024: ['wStay_min', 'w48_start_1024'],
                         1536: ['wStay_min', 'w48_start_1536']
                      },
    
    'within48_hist_full': {512: [ 0, 'w48_start_512' ],
                          1024: [ 0, 'w48_start_1024'],
                          1536: [ 0, 'w48_start_1536']
                          },
    
    'within_stay_query': {512:  ['wStay_start_512',  'wStay_end_512' ],
                          1024: ['wStay_start_1024', 'wStay_end_1024'],
                          1536: ['wStay_start_1536', 'wStay_end_1536']
                         },
    
    'within_stay_hist_icu': {512:  ['wStay_min', 'wStay_start_512' ],
                             1024: ['wStay_min', 'wStay_start_1024'],
                             1536: ['wStay_min', 'wStay_start_1536'],
                            },
    
    'within_stay_hist_full': {512:  [ 0, 'w48_start_512' ],
                              1024: [ 0, 'w48_start_1024'],
                              1536: [ 0, 'w48_start_1536'],
                             },
    }

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
class EvalDataset(Dataset):
    def __init__(self,
                 dataset_path: str,
                 data_idx_path: str,
                 seq_generator: SequencesGenerator,
                 limits_dict: dict,
                 task: str = 'y_mort',
                 main_window: str = 'within48_query', 
                 seq_length: int = 512,
                 needed_cols: list = ['subject_id', 'input_ids', 'attention_mask', 'visit_ids', 'stage_ids', 'type_ids'],
                 split: str = 'train') -> None:
        
        
        self.start_limit = limits_dict[main_window][seq_length][0]
        self.end_limit   = limits_dict[main_window][seq_length][1]
        self.task = task
        
        self.data_idx =  pl.scan_parquet(data_idx_path).collect()
        self.data_idx =  self.data_idx.filter(pl.col('split') == split)
        
        sub_ids = set(self.data_idx.get_column("subject_id").to_list())
        hf_dataset = load_from_disk(dataset_path)

        
        hf_dataset = hf_dataset.filter(
            lambda sids: [sid in sub_ids for sid in sids],
            batched=True,
            input_columns="subject_id",
        )

        # (then continue)
        self.hf_dataset = (
            hf_dataset
            .flatten_indices()
            .select_columns(needed_cols)
            .with_format("numpy", columns=needed_cols, output_all_columns=False)
        )
        
        self.seq_generator = seq_generator
        
        sids = self.hf_dataset["subject_id"]        
        self.index = defaultdict(list)
        for i, sid in enumerate(sids):
            self.index[sid].append(i)
        

        
    def __len__(self) -> int:
        return len(self.data_idx)


    
    def __getitem__(self,
                    idx: int):
        
        stay = self.data_idx[idx]
        subject_id = stay['subject_id'][0]
        label = stay[self.task][0]
       
        
        
        start = stay[self.start_limit][0]
        end = stay[self.end_limit][0]

        
        timeline_encoded = self.hf_dataset.select(self.index[subject_id])[0]
        prediction_window = {k: (v[start:end] if isinstance(v, (list, np.ndarray)) else v) for k, v in timeline_encoded.items()}
        prediction_window = seq_gen.get_overlapped_chunks(prediction_window)
        prediction_window[0]['label'] = label
        
        return prediction_window[0]

In [42]:
class EvalCollator:
    def __init__(self) -> None:
        pass


    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        chunks = self._flatten(batch)
        out = self._stack(chunks)
        
        return out


    def _flatten(self, batch) -> List[Dict]:
        out: List[Dict] = []
        for item in batch:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, (list, tuple)):
                out.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")
        if not out:
            raise ValueError("Empty batch after normalization.")
        return out

    def _stack(self, chunks: List[Dict]) -> Dict[str, torch.Tensor]:
        keys = list(chunks[0].keys())
        out = {}
        for k in keys:
            # Skip known non-numeric or variable-shaped fields
            if k in ("text_values",):  # add others you don't want to collate
                continue

            # Replace Nones with safe defaults
            seq_list = []
            for c in chunks:
                v = c[k]
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]  # 0 for ints/floats
                elif v is None:
                    # single value case (shouldn't happen for sequences, but guard anyway)
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)
        return out

In [43]:
import torch.nn as nn
class EHREmbeddings(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_size: int,
        pad_token_id: int = 0,
        type_vocab_size: int = 28,
        visit_vocab_size: int = 102,
        stage_vocab_size: int = 5,
        dropout: float = 0.1
    ):
        super().__init__()
        self.tok_emb   = nn.Embedding(vocab_size, embedding_size, padding_idx=pad_token_id)
        self.type_emb  = nn.Embedding(type_vocab_size,  embedding_size, padding_idx=pad_token_id)
        self.visit_emb = nn.Embedding(visit_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.stage_emb = nn.Embedding(stage_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.norm      = nn.LayerNorm(embedding_size)
        self.drop      = nn.Dropout(dropout)

    def encode(self, input_ids, type_ids, visit_ids, stage_ids):
        x = self.tok_emb(input_ids.long())
        x = x + self.type_emb(type_ids.long())
        x = x + self.visit_emb(visit_ids.long())
        x = x + self.stage_emb(stage_ids.long())
        return self.drop(self.norm(x))


    def forward(self, input_ids=None, 
                token_type_ids=None, 
                inputs_embeds=None,
                **kwargs):
        if inputs_embeds is not None:
            return inputs_embeds
        return self.drop(self.norm(self.tok_emb(input_ids.long())))

In [44]:
from transformers import RoFormerForSequenceClassification, RoFormerConfig, RoFormerModel
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC, BinaryAveragePrecision
import lightning as lt

import torch
import torch.nn as nn

class EvalModel(lt.LightningModule):
    def __init__(
        self,
        config,        
        ckpt_path: str = None,        
        lr: float = 2e-5,
        wd: float = 0.001,
        max_epochs: int = 100,
        dropout: float = 0.1,
        freeze: bool = False
    ):
        super().__init__()
        self.save_hyperparameters()
        self.config = config

        # ---- CHANGE 1: base model instead of sequence-classification head
        self.backbone = RoFormerModel(self.config)

        # metric buffers (unchanged)
        self.train_step_preds = []
        self.train_step_label = []
        self.val_step_preds = []
        self.val_step_label = []
        self.test_step_preds = []
        self.test_step_label = []

        # embeddings (unchanged)
        ehr_emb = EHREmbeddings(
            vocab_size=config.vocab_size, 
            embedding_size=config.embedding_size, 
            pad_token_id=config.pad_token_id,
            type_vocab_size=config.type_vocab_size, 
            visit_vocab_size=config.visit_vocab_size,
            stage_vocab_size=config.stage_vocab_size, 
            dropout=dropout
        )
        self.backbone.embeddings = ehr_emb  # RoFormerModel has .embeddings too

        # ---- CHANGE 2: single-layer binary head + BCE-with-logits
        self.classifier = nn.Linear(self.config.hidden_size, 1)
        self.criterion = nn.BCEWithLogitsLoss()

        if ckpt_path:
            self.get_pretrained_weights(model=self.backbone, ckpt_path=ckpt_path)

        if freeze:
            for param in self.backbone.parameters():
                param.requires_grad = False
            # keep only our head trainable
            for param in self.classifier.parameters():
                param.requires_grad = True

        # NOTE: we do NOT set classifier to Identity anymore, even if embed=True.
        # That previous behavior would break training for binary classification.

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs

        # metrics (unchanged)
        self.train_auroc = BinaryAUROC()
        self.train_auprc = BinaryAveragePrecision()
        self.val_auroc = BinaryAUROC()
        self.val_auprc = BinaryAveragePrecision()
        self.test_auroc = BinaryAUROC()
        self.test_auprc = BinaryAveragePrecision()

    def forward(self, 
                input_ids, 
                attention_mask, 
                type_ids, 
                visit_ids, 
                stage_ids, 
                labels=None):
        # ---- CHANGE 3: call base model, then pool, then our head; do NOT pass labels
        inputs_embeds = self.backbone.embeddings.encode(
            input_ids=input_ids,
            type_ids=type_ids,
            visit_ids=visit_ids,
            stage_ids=stage_ids
        )

        outputs = self.backbone(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            output_hidden_states=False,
            return_dict=True
        )
        last_hidden = outputs.last_hidden_state  # [B, L, H]

        # safe mean pooling using attention mask (works even without a CLS token)
#         mask = attention_mask.unsqueeze(-1).type_as(last_hidden)  # [B, L, 1]
#         summed = (last_hidden * mask).sum(dim=1)                  # [B, H]
#         lengths = mask.sum(dim=1).clamp(min=1.0)                  # [B, 1]
#         pooled = summed / lengths                                 # [B, H]
        pooled = last_hidden[:, 0, :]
        logits = self.classifier(pooled).squeeze(-1)              # [B]
        return logits

    def training_step(self, batch, batch_idx):
        logits = self.forward(input_ids=batch["input_ids"],
                              attention_mask=batch["attention_mask"],
                              type_ids=batch["type_ids"],
                              visit_ids=batch["visit_ids"],
                              stage_ids=batch["stage_ids"],
                              labels=None)

        # ---- CHANGE 4: BCEWithLogits expects float targets in {0,1}
        y = batch["label"].float().view(-1)     # [B]
        loss = self.criterion(logits, y)

        # ---- CHANGE 5: probabilities via sigmoid for metrics
        pos_score = torch.sigmoid(logits)       # [B]

        self.train_step_label.append(y)
        self.train_step_preds.append(pos_score)

        self.log("train_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_train_epoch_end(self) -> None:
        y = torch.cat(self.train_step_label)
        pos_score = torch.cat(self.train_step_preds)

        auroc = self.train_auroc(pos_score, y.long())   # metrics accept probs; cast target to long if desired
        auprc = self.train_auprc(pos_score, y.long())
        
        self.log('train_auroc', auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log('train_auprc', auprc, on_epoch=True, logger=True, prog_bar=True)

        self.train_step_label.clear()
        self.train_step_preds.clear()

    def validation_step(self, batch, batch_idx):
        logits = self.forward(input_ids=batch["input_ids"],
                              attention_mask=batch["attention_mask"],
                              type_ids=batch["type_ids"],
                              visit_ids=batch["visit_ids"],
                              stage_ids=batch["stage_ids"],
                              labels=None)

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.val_step_label.append(y)
        self.val_step_preds.append(pos_score)

        self.log("val_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss

    def on_validation_epoch_end(self,*arg, **kwargs) -> None:
        y = torch.cat(self.val_step_label)
        pos_score = torch.cat(self.val_step_preds)

        auroc = self.val_auroc(pos_score, y.long())
        auprc = self.val_auprc(pos_score, y.long())

        self.log('val_auroc', auroc, on_epoch=True, logger=True, prog_bar=True)
        self.log('val_auprc', auprc, on_epoch=True, logger=True, prog_bar=True)

        self.val_step_label.clear()
        self.val_step_preds.clear()

    def test_step(self, batch, batch_idx):
        logits = self.forward(input_ids=batch["input_ids"],
                              attention_mask=batch["attention_mask"],
                              type_ids=batch["type_ids"],
                              visit_ids=batch["visit_ids"],
                              stage_ids=batch["stage_ids"],
                              labels=None)

        y = batch["label"].float().view(-1)
        loss = self.criterion(logits, y)
        pos_score = torch.sigmoid(logits)

        self.test_step_label.append(y)
        self.test_step_preds.append(pos_score)

        self.log("test_loss", loss, prog_bar=True, on_epoch=True, logger=True)
        return loss    

    def on_test_epoch_end(self,*arg, **kwargs) -> None:
        y = torch.cat(self.test_step_label)
        pos_score = torch.cat(self.test_step_preds)

        auroc = self.test_auroc(pos_score, y.long())
        auprc = self.test_auprc(pos_score, y.long())

        self.log('test_auroc', auroc, on_epoch=True, logger=True)
        self.log('test_auprc', auprc, on_epoch=True, logger=True)

        self.test_step_label.clear()
        self.test_step_preds.clear()  

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.lr, weight_decay=self.wd)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer=optimizer,
            eta_min=0,
            T_max=self.max_epochs
        )
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}

    def get_pretrained_weights(self, model: nn.Module, ckpt_path: str) -> None:
        sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)['state_dict']

        PREFIX = "backbone.roformer."
        DROP_PREFIX = "backbone.cls."   # MLM head; not in RoFormerModel

        remapped = {}
        for k, v in sd.items():
            # skip MLM head weights entirely
            if k.startswith(DROP_PREFIX):
                continue

            if k.startswith(PREFIX):
                new_k = k[len(PREFIX):]  # strip "backbone.roformer."
            else:
                # keep as-is if already matches model keys (defensive)
                new_k = k

            remapped[new_k] = v

        missing, unexpected = model.load_state_dict(remapped, strict=False)
        print("weights loaded successfully!")
        print("missing keys:", missing)
        print('+'*50)
        print("unexpected keys:", unexpected)